# Long Short-Term Memory (LSTM)
Goal:

Understand how LSTMs solve the problems of RNNs.

Key Idea:

RNN = memory  
LSTM = controllable memory

## Why do we need LSTM?

### Problem with RNN

RNN has memory:

    h_t = f(x_t, h_{t-1})

### But memory is not stable

Problems:

1. Vanishing gradient  
2. Long-term information lost  
3. No control over what to remember or forget  

### Example

Sentence:

    "The movie was long, boring, and terrible --> sentiment negative"

RNN forgets early words.

### Core Problem

RNN memory is:

    uncontrolled

## LSTM Core Idea

Instead of:

    single memory

LSTM introduces:

    controlled memory system

Key components:

- cell state (long-term memory)
- hidden state (output)
- gates (control flow)

### Key Insight

LSTM decides:

    what to remember
    what to forget
    what to output

## LSTM Step Structure

At each time step:

Input:

    x_t (current input)
    h_{t-1} (previous state)
    c_{t-1} (long-term memory)

Output:

    h_t (new hidden state)
    c_t (updated memory)

### Main idea

Memory flows through time:

    c_{t-1} --> c_t

Controlled by gates


## LSTM Gates (The Core Mechanism)

### 1. Forget Gate

Decides:

    what to delete from memory

f_t = sigmoid(W_f [h_{t-1}, x_t])

Interpretation:

- value ~0 --> forget
- value ~1 --> keep

### 2. Input Gate

Decides:

    what new information to add


i_t = sigmoid(W_i [h_{t-1}, x_t])

### Candidate values:

~{c}_t = tanh(W_c [h_{t-1}, x_t])

### 3. Memory Update

c_t = f_t * c_{t-1} + i_t * ~{c}_t

### 4. Output Gate

Decides:

    what part of memory becomes output

o_t = sigmoid(W_o [h_{t-1}, x_t])

h_t = o_t * tanh(c_t)

### Key Insight

LSTM = gated memory control system

## Intuition for LSTM

Think of LSTM as:

a scientist taking notes

At each step:

- Forget irrelevant info
- Add new observations
- Keep important context


Example (text):

"The stock price rose yesterday but fell today"

LSTM learns:

- yesterday info less important
- today info more important

### Key Insight

Memory is actively managed, not passively stored


In [ ]:
# SEQUENCE DATA

import numpy as np

def generate_sequence_data(n_samples=1000, seq_length=5):

    X, y = [], []

    for _ in range(n_samples):
        seq = np.random.randint(0, 10, size=seq_length)
        
        X.append(seq[:-1])
        y.append(seq[-1])

    return np.array(X), np.array(y)

X, y = generate_sequence_data()

print(X.shape, y.shape)

In [ ]:
import torch
import torch.nn as nn

X_torch = torch.tensor(X, dtype=torch.float32)
y_torch = torch.tensor(y, dtype=torch.long)

In [ ]:
# LSTM MODEL

class SimpleLSTM(nn.Module):

    def __init__(self, input_size=1, hidden_size=32, output_size=10):
        super().__init__()

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        # output layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):

        x = x.unsqueeze(-1)

        out, (h_n, c_n) = self.lstm(x)

        # take last time step
        out = out[:, -1, :]

        out = self.fc(out)

        return out

model = SimpleLSTM()

In [ ]:
# TRAINING

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(50):

    optimizer.zero_grad()

    outputs = model(X_torch)

    loss = criterion(outputs, y_torch)

    loss.backward()

    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: {loss.item():.4f}")


In [ ]:
preds = model(X_torch).argmax(dim=1)

accuracy = (preds == y_torch).float().mean()

print("Accuracy:", accuracy.item())

## Why LSTM Works Better Than RNN

RNN:

- memory overwritten quickly
- gradients vanish

LSTM:

- memory flows through cell state
- gates control updates
- stable gradient flo

### Result

LSTM can:

- learn long-term dependencies
- retain important information
- ignore noise